# ODI to Databricks Migration: TRG_DEP_INSERT

**Conversion Timestamp:** 2024-07-30 12:00:00

**Description:** This notebook converts an ODI INSERT statement for the `TRG_DEP` table from Oracle SQL to Databricks Spark SQL. It includes common ETL parameter widgets and schema/syntax conversions.

In [ ]:
# SCEN_TASK_NO {10}: Initialize ETL Parameters
dbutils.widgets.text("ETL_JOB_TYPE", "", "ETL Job Type")
dbutils.widgets.text("DATASOURCE_NUM_ID", "1", "Datasource Number ID")
dbutils.widgets.text("ETL_PROC_WID", "-1", "ETL Process WID")
dbutils.widgets.text("ODI_SESS_NO", "-1", "ODI Session Number")

## ETL Parameters

In [ ]:
display(spark.sql("""
  SELECT
    '${ETL_JOB_TYPE}' AS ETL_JOB_TYPE,
    ${DATASOURCE_NUM_ID} AS DATASOURCE_NUM_ID,
    ${ETL_PROC_WID} AS ETL_PROC_WID,
    '${ODI_SESS_NO}' AS ODI_SESS_NO
"""))

## Target Table: `workspace.hr.trg_dep`

In [ ]:
%sql
-- SCEN_TASK_NO {20}: Create or Replace Target Table DDL
-- Inferring DDL based on common HR schema for DEPARTMENTS
CREATE TABLE IF NOT EXISTS workspace.hr.trg_dep (
    department_id   BIGINT,
    department_name STRING,
    manager_id      BIGINT,
    location_id     BIGINT
) USING DELTA;

In [ ]:
%sql
-- SCEN_TASK_NO {30}: Insert into Target Table
-- Converted: Removed Oracle hints /*+ APPEND PARALLEL */.
-- Converted: Schema and table names to lowercase with 'workspace.' prefix.
INSERT INTO workspace.hr.trg_dep
(
  department_id,
  department_name,
  manager_id,
  location_id
)
SELECT
  departments.department_id,
  departments.department_name,
  departments.manager_id,
  departments.location_id
FROM
  workspace.hr.departments AS departments;

## Validation

In [ ]:
%sql
-- Validate the number of records inserted
SELECT COUNT(*) AS total_records_in_trg_dep
FROM workspace.hr.trg_dep;

In [ ]:
%sql
-- Sample recently inserted records
SELECT *
FROM workspace.hr.trg_dep
LIMIT 10;

## Conversion Notes and Manual Actions Required

1.  **Inferred DDL:** The `CREATE TABLE IF NOT EXISTS workspace.hr.trg_dep` statement was inferred based on common Oracle HR schema types. It is crucial to verify this DDL against the actual source Oracle DDL for `HR.TRG_DEP` and `HR.DEPARTMENTS` to ensure correct data types (especially for `NUMBER` precision/scale) and column order if not explicitly specified in the `INSERT` list.
2.  **Oracle Hints:** Oracle-specific hints like `/*+ APPEND PARALLEL */` have been removed as they are not applicable to Databricks Delta Lake and Spark SQL.
3.  **Schema and Table Naming:** All schema and table names have been converted to lowercase and prefixed with `workspace.` (e.g., `HR.TRG_DEP` -> `workspace.hr.trg_dep`).
4.  **Parameter Usage:** No specific ODI parameters (`#GLOBAL` or `#SCHEMA`) were found in the provided SQL. Generic ETL widgets (`ETL_JOB_TYPE`, `DATASOURCE_NUM_ID`, `ETL_PROC_WID`, `ODI_SESS_NO`) are included as per the standard notebook structure but are not used in the core `INSERT` statement in this specific conversion.
5.  **Performance Optimization:** For large tables, consider adding `OPTIMIZE` and `ZORDER BY` clauses to the target table after the `INSERT` operation for improved query performance, based on common query patterns. E.g.:
    ```sql
    -- Disable ZORDER stats check to prevent DELTA_ZORDERING_ON_COLUMN_WITHOUT_STATS
    SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;
    OPTIMIZE workspace.hr.trg_dep ZORDER BY (department_id);
    ```